# Day 019 | Pandas — Series & DataFrames

## 100 Days of ML for Geotechnical Engineering
**Phase:** Data Science Essentials | **Author:** Ripon Chandra Malo | **Date:** YYYY-MM-DD

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| `pd.Series` | 1D labeled array | SPT N-values at each depth |
| `pd.DataFrame` | 2D labeled table | Complete borehole log |
| `read_csv` / `to_csv` | Read/write tabular files | Load borehole data from CSV |
| `[]`, `.loc[]`, `.iloc[]` | Select rows and columns | Get all sand layers, depth 6-12m |
| `.info()`, `.describe()` | Quick data summary | Check data types, missing values |
| Add/remove columns | Compute derived values | Calculate N60, overburden pressure |
| Sorting and ranking | Order data | Sort layers by depth, rank by strength |

---

## Why Pandas for Geotechnical Engineering?

Geotechnical data is **tabular** — rows are depths or samples, columns are properties:

```
Depth(m)  N_value  Soil_Type    Water_Content  Unit_Weight
  1.5       4      Fill          22.5           16.8
  3.0       6      Silty Sand    18.3           17.2
  4.5       8      Silty Sand    19.1           17.5
  ...
```

Pandas is **the** tool for this kind of data. It gives you:
- Named columns (no more remembering that column 3 is water content)
- Powerful filtering (`df[df.soil_type == 'Sand']`)
- Built-in statistics (`.describe()` gives mean, std, min, max)
- CSV/Excel import with one line

Every ML pipeline starts with Pandas.

---

In [ ]:
# ─────────────────────────────────────────────────────
# SETUP: Import libraries
# ─────────────────────────────────────────────────────
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("NumPy version: ", np.__version__)
print("Ready for geotechnical data analysis!")

---

## Part 1: Series — The 1D Building Block

A `pd.Series` is like a NumPy array with **labels** (an index). Think of it as one column of a borehole log.

```python
# NumPy: just values
np.array([4, 6, 8, 12, 15])

# Pandas Series: values + labels
pd.Series([4, 6, 8, 12, 15], index=[1.5, 3.0, 4.5, 6.0, 7.5], name='N_value')
```

The labels make the data **self-documenting** — you know each N-value belongs to a specific depth.

In [ ]:
# ─────────────────────────────────────────────────────
# SERIES: SPT N-values with depth labels
# ─────────────────────────────────────────────────────

# Create a Series of SPT N-values, indexed by depth
depths = [1.5, 3.0, 4.5, 6.0, 7.5, 9.0, 10.5, 12.0]
n_values = [4, 6, 8, 12, 15, 18, 22, 14]

spt = pd.Series(n_values, index=depths, name='N_value')
spt.index.name = 'depth_m'

print("SPT N-value Series:")
print(spt)
print(f"\nType: {type(spt)}")
print(f"Shape: {spt.shape}")
print(f"Dtype: {spt.dtype}")

In [ ]:
# ─────────────────────────────────────────────────────
# SERIES: Indexing and basic operations
# ─────────────────────────────────────────────────────

# Access by label (depth)
print("N-value at 6.0m:", spt[6.0])
print("N-values at 3.0-6.0m:")
print(spt[3.0:6.0])  # Inclusive on both ends for label-based slicing!

# Basic statistics
print(f"\nStatistics:")
print(f"  Mean N:   {spt.mean():.1f}")
print(f"  Std N:    {spt.std():.1f}")
print(f"  Min N:    {spt.min()} at {spt.idxmin()}m")
print(f"  Max N:    {spt.max()} at {spt.idxmax()}m")

# Vectorized operations — compute shear wave velocity
# Vs = 97 * N^0.314 (Imai & Tonouchi, 1982)
Vs = 97 * spt ** 0.314
Vs.name = 'Vs_m/s'
print(f"\nShear wave velocity (Vs = 97 * N^0.314):")
print(Vs.round(1))

# Boolean filtering — find loose layers
loose = spt[spt < 10]
print(f"\nLoose layers (N < 10):")
print(loose)

---

## Part 2: DataFrame — The Complete Borehole Log

A `pd.DataFrame` is a 2D table — multiple Series sharing the same index. It's perfect for borehole logs where each row is a depth and each column is a property.

Three ways to create a DataFrame:
1. **From a dictionary** — keys become column names
2. **From a list of lists** — specify column names separately
3. **From a CSV file** — most common in practice

In [ ]:
# ─────────────────────────────────────────────────────
# DATAFRAME: Create from dictionary (borehole log)
# ─────────────────────────────────────────────────────

borehole_data = {
    'depth': [1.5, 3.0, 4.5, 6.0, 7.5, 9.0, 10.5, 12.0],
    'N_value': [4, 6, 8, 12, 15, 18, 22, 14],
    'soil_type': ['Fill', 'Silty Sand', 'Silty Sand', 'Sand',
                  'Sand', 'Clayey Sand', 'Sand', 'Silty Clay'],
    'water_content': [22.5, 18.3, 19.1, 15.6, 14.2, 20.8, 13.5, 28.4],
    'unit_weight': [16.8, 17.2, 17.5, 18.0, 18.3, 17.8, 18.5, 17.0],
}

df = pd.DataFrame(borehole_data)

print("Borehole Log DataFrame:")
print(df)
print(f"\nShape: {df.shape} ({df.shape[0]} rows, {df.shape[1]} columns)")
print(f"Columns: {list(df.columns)}")

In [ ]:
# ─────────────────────────────────────────────────────
# DATAFRAME: Basic info — .info(), .describe(), .dtypes
# ─────────────────────────────────────────────────────

print("=" * 50)
print(".info() — Data types and memory usage:")
print("=" * 50)
df.info()

print(f"\n{'=' * 50}")
print(".describe() — Summary statistics:")
print("=" * 50)
print(df.describe().round(2))

print(f"\n{'=' * 50}")
print(".dtypes — Column data types:")
print("=" * 50)
print(df.dtypes)

---

## Part 3: Reading and Writing CSV Files

In practice, you rarely type in data by hand. You load it from CSV files exported from gINT, AGS, or lab software.

```python
df = pd.read_csv('borehole_log.csv')   # Load
df.to_csv('output.csv', index=False)    # Save
```

In [ ]:
# ─────────────────────────────────────────────────────
# READ CSV: Load our borehole log
# ─────────────────────────────────────────────────────

# Load the 20-row borehole log CSV
bh = pd.read_csv('borehole_log.csv')

print(f"Loaded borehole_log.csv: {bh.shape[0]} rows, {bh.shape[1]} columns")
print(f"Columns: {list(bh.columns)}\n")

# First 5 rows
print("First 5 rows (.head()):")
print(bh.head())

# Last 5 rows
print(f"\nLast 5 rows (.tail()):")
print(bh.tail())

In [ ]:
# ─────────────────────────────────────────────────────
# CSV: Quick overview of the full dataset
# ─────────────────────────────────────────────────────

print("Full dataset info:")
bh.info()

print(f"\nSummary statistics:")
print(bh.describe().round(2))

print(f"\nSoil types present:")
print(bh['soil_type'].value_counts())

---

## Part 4: Selecting Columns and Rows

Three selection methods — know when to use each:

| Method | Syntax | Selects By | Example |
|--------|--------|-----------|--------|
| `[]` | `df['col']` or `df[mask]` | Column name or boolean mask | `bh['N_value']`, `bh[bh.depth > 10]` |
| `.loc[]` | `df.loc[rows, cols]` | **Labels** | `bh.loc[0:4, 'depth':'N_value']` |
| `.iloc[]` | `df.iloc[rows, cols]` | **Integer position** | `bh.iloc[0:5, 0:2]` |

**Rule of thumb:** Use `[]` for quick grabs, `.loc` when you need specific rows AND columns by name, `.iloc` when you need rows by position number.

In [ ]:
# ─────────────────────────────────────────────────────
# SELECTING: Columns — single, multiple, and conditions
# ─────────────────────────────────────────────────────

# Single column → returns a Series
n_vals = bh['N_value']
print("Single column (Series):")
print(n_vals.head())
print(f"Type: {type(n_vals)}\n")

# Multiple columns → returns a DataFrame
subset = bh[['depth', 'N_value', 'soil_type']]
print("Multiple columns (DataFrame):")
print(subset.head())
print(f"Type: {type(subset)}\n")

# Boolean filtering — get sand layers only
sand_layers = bh[bh['soil_type'] == 'Sand']
print(f"Sand layers only ({len(sand_layers)} rows):")
print(sand_layers[['depth', 'N_value', 'soil_type']])

# Multiple conditions — sand with N > 20
strong_sand = bh[(bh['soil_type'] == 'Sand') & (bh['N_value'] > 20)]
print(f"\nSand with N > 20 ({len(strong_sand)} rows):")
print(strong_sand[['depth', 'N_value', 'soil_type']])

In [ ]:
# ─────────────────────────────────────────────────────
# SELECTING: .loc[] and .iloc[]
# ─────────────────────────────────────────────────────

# .loc — label-based selection
# Get rows 0-4, columns 'depth' through 'soil_type'
print(".loc[0:4, 'depth':'soil_type'] — by label (inclusive both ends):")
print(bh.loc[0:4, 'depth':'soil_type'])

# .iloc — integer position-based selection
# Get rows 0-4, columns 0-2 (exclusive end!)
print(f"\n.iloc[0:5, 0:3] — by position (exclusive end):")
print(bh.iloc[0:5, 0:3])

# .loc with condition — specific columns for deep layers
print(f"\nDeep layers (>20m) — depth, N, and soil:")
deep = bh.loc[bh['depth'] > 20, ['depth', 'N_value', 'soil_type']]
print(deep)

# .iloc — last 3 rows, first 3 columns
print(f"\nLast 3 rows, first 3 columns (iloc):")
print(bh.iloc[-3:, :3])

---

## Part 5: Adding and Removing Columns

In geotechnical engineering, raw field data is just the starting point. We need to **compute derived values**:

- **N60** (corrected blow count) = N_field * correction_factors
- **Overburden pressure** ($\sigma_v$) = cumulative unit_weight * depth
- **Relative density** from N-value
- **Shear wave velocity** from empirical correlations

In [ ]:
# ─────────────────────────────────────────────────────
# ADD COLUMNS: Compute N60 and overburden pressure
# ─────────────────────────────────────────────────────

# Work on a copy to keep the original clean
bh_calc = bh.copy()

# --- N60 correction ---
# N60 = N_field * Em/60 (simplified: assume Em = 72% for safety hammer)
hammer_efficiency = 0.72   # Safety hammer
correction_ratio = hammer_efficiency / 0.60
bh_calc['N60'] = (bh_calc['N_value'] * correction_ratio).round(0).astype(int)

print(f"Hammer efficiency: {hammer_efficiency*100:.0f}%")
print(f"Correction: N60 = N_field * {correction_ratio:.2f}\n")

# --- Overburden pressure ---
# Simplified: sigma_v = unit_weight * depth (kPa)
# (In reality, you'd integrate layer by layer)
bh_calc['sigma_v'] = (bh_calc['unit_weight'] * bh_calc['depth']).round(1)

# --- Effective overburden (assuming water table at 3m) ---
water_table = 3.0  # meters below ground
gamma_w = 9.81     # kN/m3
bh_calc['u'] = np.where(
    bh_calc['depth'] > water_table,
    gamma_w * (bh_calc['depth'] - water_table),
    0
).round(1)
bh_calc['sigma_v_eff'] = (bh_calc['sigma_v'] - bh_calc['u']).round(1)

print("Computed columns added:")
print(bh_calc[['depth', 'N_value', 'N60', 'sigma_v', 'u', 'sigma_v_eff']].head(10))

In [ ]:
# ─────────────────────────────────────────────────────
# ADD MORE: Consistency classification
# ─────────────────────────────────────────────────────

# Classify based on N60 (for granular soils)
def classify_density(n60):
    """Classify relative density from N60 (granular soils)."""
    if n60 < 4:   return 'Very Loose'
    if n60 < 10:  return 'Loose'
    if n60 < 30:  return 'Medium Dense'
    if n60 < 50:  return 'Dense'
    return 'Very Dense'

bh_calc['density_class'] = bh_calc['N60'].apply(classify_density)

print("With density classification:")
print(bh_calc[['depth', 'soil_type', 'N60', 'density_class']].to_string())

print(f"\nDensity distribution:")
print(bh_calc['density_class'].value_counts())

# --- Remove a column ---
# Drop the pore pressure column (just to show how)
bh_dropped = bh_calc.drop(columns=['u'])
print(f"\nAfter dropping 'u' column: {list(bh_dropped.columns)}")

---

## Part 6: Sorting and Ranking

Sorting organizes data for presentation and analysis. Ranking assigns ordinal positions.

```python
df.sort_values('column')           # Sort ascending
df.sort_values('column', ascending=False)  # Sort descending
df['rank'] = df['column'].rank()   # Assign ranks
```

In [ ]:
# ─────────────────────────────────────────────────────
# SORTING: By depth, by N-value, by multiple columns
# ─────────────────────────────────────────────────────

# Sort by N60 descending — find the strongest layers
strongest = bh_calc.sort_values('N60', ascending=False)
print("Layers sorted by N60 (strongest first):")
print(strongest[['depth', 'soil_type', 'N60', 'density_class']].head(8).to_string())

# Sort by soil type, then by depth within each type
by_soil = bh_calc.sort_values(['soil_type', 'depth'])
print(f"\nLayers grouped by soil type:")
print(by_soil[['depth', 'soil_type', 'N60']].to_string())

In [ ]:
# ─────────────────────────────────────────────────────
# RANKING: Rank layers by strength
# ─────────────────────────────────────────────────────

# Rank by N60 (1 = weakest, 20 = strongest)
bh_calc['strength_rank'] = bh_calc['N60'].rank().astype(int)

# Rank by water content (1 = driest, 20 = wettest)
bh_calc['moisture_rank'] = bh_calc['water_content'].rank().astype(int)

print("Rankings:")
print(bh_calc[['depth', 'soil_type', 'N60', 'strength_rank',
               'water_content', 'moisture_rank']].to_string())

# Weakest layer
weakest_idx = bh_calc['N60'].idxmin()
print(f"\nWeakest layer: depth = {bh_calc.loc[weakest_idx, 'depth']}m, "
      f"N60 = {bh_calc.loc[weakest_idx, 'N60']}, "
      f"soil = {bh_calc.loc[weakest_idx, 'soil_type']}")

# Strongest layer
strongest_idx = bh_calc['N60'].idxmax()
print(f"Strongest layer: depth = {bh_calc.loc[strongest_idx, 'depth']}m, "
      f"N60 = {bh_calc.loc[strongest_idx, 'N60']}, "
      f"soil = {bh_calc.loc[strongest_idx, 'soil_type']}")

---

## Part 7: Complete Application — Borehole Data Analysis Pipeline

Putting it all together: load borehole data, compute derived columns (N60, overburden stress, shear-wave velocity), and generate a comprehensive summary.

```
borehole_log.csv → Load → Compute N60, sigma_v, Vs → Summary statistics → Save
```

In [ ]:
# ═══════════════════════════════════════════════════════
# COMPLETE APPLICATION: Borehole Data Analysis
# ═══════════════════════════════════════════════════════

# --- Step 1: Load data ---
bh_full = pd.read_csv('borehole_log.csv')
print(f"Step 1: Loaded {bh_full.shape[0]} layers from borehole_log.csv")
print(f"        Depth range: {bh_full['depth'].min()}m to {bh_full['depth'].max()}m")
print(f"        Soil types:  {bh_full['soil_type'].nunique()} unique")

# --- Step 2: Compute N60 ---
Ce = 0.72 / 0.60     # Energy ratio (safety hammer)
Cb = 1.0              # Borehole diameter correction (65-115mm)
Cs = 1.0              # Sampler correction (standard)
Cr_values = np.where(bh_full['depth'] < 3, 0.75,
            np.where(bh_full['depth'] < 4, 0.80,
            np.where(bh_full['depth'] < 6, 0.85,
            np.where(bh_full['depth'] < 10, 0.95, 1.00))))
bh_full['Cr'] = Cr_values
bh_full['N60'] = (bh_full['N_value'] * Ce * Cb * Cs * Cr_values).round(0).astype(int)
print(f"\nStep 2: N60 computed (Ce={Ce:.2f}, Cr varies with depth)")

# --- Step 3: Overburden stress ---
gamma_w = 9.81
wt_depth = 3.0  # water table depth
bh_full['sigma_v'] = (bh_full['unit_weight'] * bh_full['depth']).round(1)
bh_full['u'] = np.maximum(0, gamma_w * (bh_full['depth'] - wt_depth)).round(1)
bh_full['sigma_v_eff'] = (bh_full['sigma_v'] - bh_full['u']).round(1)
print(f"Step 3: Overburden stresses computed (WT at {wt_depth}m)")

# --- Step 4: Shear wave velocity ---
# Vs = 97 * N60^0.314 (Imai & Tonouchi, 1982)
bh_full['Vs'] = (97 * bh_full['N60'] ** 0.314).round(1)
print(f"Step 4: Vs computed (Imai & Tonouchi, 1982)")

# --- Display results ---
print(f"\n{'='*80}")
print("BOREHOLE ANALYSIS RESULTS")
print(f"{'='*80}")
display_cols = ['depth', 'soil_type', 'N_value', 'N60', 'sigma_v', 'sigma_v_eff', 'Vs']
print(bh_full[display_cols].to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════
# Step 5: Summary statistics by soil type
# ═══════════════════════════════════════════════════════

print("SUMMARY BY SOIL TYPE")
print("=" * 60)

soil_summary = bh_full.groupby('soil_type').agg(
    count=('N60', 'count'),
    N60_mean=('N60', 'mean'),
    N60_min=('N60', 'min'),
    N60_max=('N60', 'max'),
    Vs_mean=('Vs', 'mean'),
    wc_mean=('water_content', 'mean'),
    fines_mean=('fines_content', 'mean'),
).round(1)

print(soil_summary.to_string())

# Overall statistics
print(f"\nOVERALL STATISTICS")
print("=" * 60)
print(f"Total depth:         {bh_full['depth'].max()} m")
print(f"Number of layers:    {len(bh_full)}")
print(f"Average N60:         {bh_full['N60'].mean():.1f}")
print(f"Average Vs:          {bh_full['Vs'].mean():.1f} m/s")
print(f"Vs30 (approx):       {bh_full.loc[bh_full['depth'] <= 30, 'Vs'].mean():.1f} m/s")

# --- Save results ---
output_cols = ['depth', 'soil_type', 'N_value', 'N60', 'water_content',
               'unit_weight', 'fines_content', 'sigma_v', 'sigma_v_eff', 'Vs']
bh_full[output_cols].to_csv('borehole_analysis.csv', index=False)
print(f"\nResults saved to borehole_analysis.csv")

---

## Practice Exercises

### Exercise 1: Filter and summarize
Find all layers where `fines_content > 30%`. How many are there? What is the average water content of these fine-grained layers compared to the coarse-grained ones?

### Exercise 2: Depth-weighted average
Compute the depth-weighted average N60 for the top 15m of the borehole. (Hint: each layer represents 1.5m thickness.)

### Exercise 3: Liquefaction screening
Create a new column `liq_susceptible` that is `True` if ALL of these conditions are met:
- Depth is below water table (> 3.0m)
- N60 < 25
- Fines content < 35%

How many layers are potentially liquefiable?

In [ ]:
# EXERCISE 1: Filter and summarize
# fine_layers = bh_full[bh_full['fines_content'] > 30]
# coarse_layers = bh_full[bh_full['fines_content'] <= 30]
# print(f"Fine-grained: {len(fine_layers)} layers, avg wc = {fine_layers['water_content'].mean():.1f}%")
# print(f"Coarse-grained: {len(coarse_layers)} layers, avg wc = {coarse_layers['water_content'].mean():.1f}%")

In [ ]:
# EXERCISE 2: Depth-weighted average N60
# top_15 = bh_full[bh_full['depth'] <= 15.0]
# thickness = 1.5  # each layer is 1.5m
# weighted_avg = (top_15['N60'] * thickness).sum() / (len(top_15) * thickness)
# print(f"Weighted average N60 (top 15m): {weighted_avg:.1f}")

In [ ]:
# EXERCISE 3: Liquefaction screening
# bh_full['liq_susceptible'] = (
#     (bh_full['depth'] > 3.0) &
#     (bh_full['N60'] < 25) &
#     (bh_full['fines_content'] < 35)
# )
# print(bh_full[['depth', 'soil_type', 'N60', 'fines_content', 'liq_susceptible']])
# print(f"\nLiquefiable layers: {bh_full['liq_susceptible'].sum()}")

---

## Key Takeaways

1. **`pd.Series` is a labeled 1D array, `pd.DataFrame` is a labeled 2D table.** Labels (index + column names) make geotechnical data self-documenting — you always know what depth and property you're looking at.

2. **`read_csv` loads tabular data in one line.** Combined with `.info()` and `.describe()`, you can understand a new dataset in seconds.

3. **Use `[]` for quick column/row grabs, `.loc` for label-based selection, `.iloc` for position-based.** Boolean masks (`df[df.N60 > 30]`) are the most powerful filtering tool.

4. **Adding computed columns is straightforward:** `df['N60'] = df['N_value'] * correction`. Vectorized operations make Pandas fast even for large datasets.

5. **`.sort_values()` and `.rank()` help organize results** for engineering reports — sort layers by strength to find the weakest zone, rank by properties for comparison.

---

## How This Connects to ML

| Today | ML Equivalent |
|-------|---------------|
| `pd.read_csv` | Loading training data |
| `.describe()` | Exploratory Data Analysis (EDA) |
| Boolean filtering | Removing outliers, selecting subsets |
| Adding computed columns | Feature engineering (N60, Vs from N) |
| `.info()` / `.dtypes` | Checking for type mismatches before model training |
| Sorting / ranking | Understanding feature importance rankings |

**Pandas is the gateway to ML.** Every scikit-learn model, every TensorFlow pipeline, every data visualization starts with a DataFrame. Mastering Series and DataFrames today means smoother ML work ahead.

---

*Day 019 of 100 — We now speak the language of tabular data!*